# 01 — Extração: SQL Server → MinIO `landing-zone`

Lê todas as tabelas do SQL Server e salva cada uma como arquivo **CSV** no bucket `landing-zone`.

| Origem | Destino |
|---|---|
| `seguradora.<tabela>` | `landing-zone/<tabela>/<tabela>.csv` |

In [1]:
import io
import os
import csv
import pyodbc
import boto3
from botocore.client import Config
from dotenv import load_dotenv

load_dotenv()

True

## Conexões

In [2]:
conn_str = (
    f"DRIVER={{ODBC Driver 18 for SQL Server}};"
    f"SERVER={os.getenv('SQLSERVER_HOST')},{os.getenv('SQLSERVER_PORT')};"
    f"DATABASE={os.getenv('SQLSERVER_DB')};"
    f"UID={os.getenv('SQLSERVER_USER')};"
    f"PWD={os.getenv('SQLSERVER_PASSWORD')};"
    f"TrustServerCertificate=yes;"
)

conn   = pyodbc.connect(conn_str)
cursor = conn.cursor()
print("SQL Server:", os.getenv("SQLSERVER_DB"))

s3 = boto3.client(
    "s3",
    endpoint_url=os.getenv("MINIO_ENDPOINT"),
    aws_access_key_id=os.getenv("MINIO_ACCESS_KEY"),
    aws_secret_access_key=os.getenv("MINIO_SECRET_KEY"),
    config=Config(signature_version="s3v4"),
)
LANDING = os.getenv("MINIO_LANDING_BUCKET")
print("MinIO bucket:", LANDING)

SQL Server: seguradora
MinIO bucket: landing-zone


## Criação do bucket `landing-zone`

In [3]:
existing = [b["Name"] for b in s3.list_buckets()["Buckets"]]
if LANDING not in existing:
    s3.create_bucket(Bucket=LANDING)
    print(f"Bucket '{LANDING}' criado.")
else:
    print(f"Bucket '{LANDING}' já existe.")

Bucket 'landing-zone' já existe.


## Extração e upload

In [4]:
# Lista todas as tabelas do banco
cursor.execute(
    "SELECT TABLE_NAME FROM INFORMATION_SCHEMA.TABLES "
    "WHERE TABLE_TYPE = 'BASE TABLE' ORDER BY TABLE_NAME"
)
TABLES = [row[0] for row in cursor.fetchall()]
print("Tabelas encontradas:", TABLES)

for table in TABLES:
    cursor.execute(f"SELECT * FROM {table}")
    rows   = cursor.fetchall()
    cols   = [desc[0] for desc in cursor.description]

    # Serializa para CSV em memória
    buf = io.StringIO()
    writer = csv.writer(buf)
    writer.writerow(cols)
    writer.writerows(rows)
    payload = buf.getvalue().encode("utf-8")

    key = f"{table}/{table}.csv"
    s3.put_object(
        Bucket=LANDING,
        Key=key,
        Body=io.BytesIO(payload),
        ContentType="text/csv",
    )
    print(f"  {table}: {len(rows)} docs → s3://{LANDING}/{key}")

conn.close()
print("\nExtração concluída.")

Tabelas encontradas: ['apolice', 'carro', 'cliente', 'endereco', 'estado', 'marca', 'modelo', 'municipio', 'regiao', 'sinistro', 'telefone']
  apolice: 10 docs → s3://landing-zone/apolice/apolice.csv
  carro: 10 docs → s3://landing-zone/carro/carro.csv
  cliente: 10 docs → s3://landing-zone/cliente/cliente.csv
  endereco: 10 docs → s3://landing-zone/endereco/endereco.csv
  estado: 15 docs → s3://landing-zone/estado/estado.csv
  marca: 8 docs → s3://landing-zone/marca/marca.csv
  modelo: 15 docs → s3://landing-zone/modelo/modelo.csv
  municipio: 10 docs → s3://landing-zone/municipio/municipio.csv
  regiao: 5 docs → s3://landing-zone/regiao/regiao.csv
  sinistro: 8 docs → s3://landing-zone/sinistro/sinistro.csv
  telefone: 10 docs → s3://landing-zone/telefone/telefone.csv

Extração concluída.


## Verificação — listar arquivos no `landing-zone`

In [5]:
response = s3.list_objects_v2(Bucket=LANDING)
print(f'Arquivos em "{LANDING}":')
for obj in response.get("Contents", []):
    size_kb = obj["Size"] / 1024
    print(f'  {obj["Key"]}  ({size_kb:.1f} KB)')

Arquivos em "landing-zone":
  apolice/apolice.csv  (0.6 KB)
  carro/carro.csv  (0.3 KB)
  cliente/cliente.csv  (0.7 KB)
  endereco/endereco.csv  (0.5 KB)
  estado/estado.csv  (0.3 KB)
  marca/marca.csv  (0.2 KB)
  modelo/modelo.csv  (0.3 KB)
  municipio/municipio.csv  (0.2 KB)
  regiao/regiao.csv  (0.1 KB)
  sinistro/sinistro.csv  (0.5 KB)
  telefone/telefone.csv  (0.3 KB)
